# Stage-by-stage validation of the decision framework

This notebook accompanies Section IV-G-1 of the article. It imports the program files in this folder and runs the validation one stage at a time, so you can inspect each intermediate result:

- the data;
- the validation/test split;
- the candidate ranking and the selected configuration;
- the test metrics;
- the significance tests.

| Part | Program file used | Reproduces |
|---|---|---|
| A | `synthetic_ofdm_validation.py` | Synthetic OFDM held-out test performance and its p-values |
| B | `measured_csi_validation.py` | Real measured-channel held-out test results and its p-values |
| all | `decision_framework_core.py` | Shared receiver chain and configuration selection |




## Setup

In [1]:
import platform
import sys
from dataclasses import asdict, replace
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

ROOT = Path.cwd()
if not (ROOT / "decision_framework_core.py").exists():
    raise FileNotFoundError("Start the notebook from the folder that contains decision_framework_core.py.")
sys.path.insert(0, str(ROOT))

import decision_framework_core as core
import synthetic_ofdm_validation as syn
import measured_csi_validation as mea

print("Python    ", platform.python_version())
for pkg in ["numpy", "scipy", "pandas", "pywavelets"]:
    print(f"{pkg:<10}", version(pkg))

Python     3.14.0
numpy      2.3.4
scipy      1.16.3
pandas     2.3.3
pywavelets 1.9.0


---
## Part A: synthetic OFDM link

### A1. Simulate the frames

The link has:

- 128 subcarriers with a 32-sample cyclic prefix;
- comb pilots on every 4th subcarrier, giving 32 pilots and 96 data subcarriers;
- QPSK modulation;
- a 12-tap Rayleigh channel with an exponential power-delay profile.

One seeded random stream (seed 7) generates all 800 frames: 2 noise regimes × 5 SNR values × 80 frames.

In [2]:
fb, regime_of = syn.simulate_dataset()
print(f"{len(fb.snr_db)} frames, {fb.n_sc} subcarriers "
      f"({len(fb.pilot_idx)} pilots, {len(fb.data_idx)} data)")
pd.crosstab(pd.Series(regime_of, name="noise regime"), pd.Series(fb.snr_db, name="SNR (dB)"))

800 frames, 128 subcarriers (32 pilots, 96 data)


SNR (dB),0.0,5.0,10.0,15.0,20.0
noise regime,,,,,
awgn,80,80,80,80,80
bursty_impulsive,80,80,80,80,80


### A2. Split into validation and test sets

The split is stratified by noise regime and SNR. Within each (regime, SNR) cell, 40 % of the frames go to the validation set (seed 10) and the remaining 60 % are held out for testing. The configuration selection never sees the test frames.

In [3]:
splits = syn.split_frames(fb, regime_of)
pd.DataFrame([{"noise regime": label,
               "validation frames": len(splits[reg][0].snr_db),
               "test frames": len(splits[reg][1].snr_db),
               "frames in both": len(np.intersect1d(splits[reg][0].frame_id, splits[reg][1].frame_id))}
              for reg, label in syn.REGIMES.items()])

,noise regime,validation frames,test frames,frames in both
0,AWGN,160,240,0
1,Impulsive,160,240,0


### A3. Candidate configurations

The search is centred on the band-aligned depth J_band = ⌈log₂((f_s/2)/B)⌉, where f_s = 30.72 MHz and B = 5 MHz. The levels searched are J_band − 1, J_band and J_band + 1.

The candidate set contains every admissible combination of:

- transform: DWT, SWT, WPT;
- mother wavelet: haar, db4, sym4, dmey;
- level;
- threshold rule: sqtwolog, rigrsure;
- shrinkage: soft, garrote;
- threshold scale: 0.5, 1.0;
- complex handling: I/Q separately (component) or magnitude–phase.

A combination is admissible only if its level does not exceed the maximum level allowed for that wavelet.

In [4]:
print(f"J_band = {syn.J_BAND}, levels searched = {syn.LEVELS}")
grid13 = pd.DataFrame([asdict(c) for c in syn.GRID])
print(len(grid13), "candidate configurations. Number per transform and wavelet:")
grid13.groupby(["transform", "wavelet"]).size().unstack(fill_value=0)

J_band = 2, levels searched = [1, 2, 3]
512 candidate configurations. Number per transform and wavelet:


wavelet,db4,dmey,haar,sym4
transform,,,,
DWT,48,16,48,48
SWT,48,48,48,48
WPT,48,16,48,48


### A4. Select the configuration on the validation set

Selection uses the validation set only and runs in two steps:

1. Every candidate is scored with a zero-forcing receiver and ranked by BER, then EVM, then NMSE.
2. The delay-support length of the winning candidate is then tuned by NMSE (0 means off).

On the same validation set, the number of DFT taps (DFT baseline) and the clipping factor (clipping baseline, used only in the Holm family) are tuned by BER, then EVM, then NMSE.

This step takes a few seconds.

In [5]:
selections13 = {}
for reg, label in syn.REGIMES.items():
    selections13[reg] = s = syn.select_on_validation(splits[reg][0])
    print(f"[{label}] framework: {s['framework'].label()} | DFT taps: {s['dft_taps']} | clip factor: {s['clip_factor']}")

[AWGN] framework: WPT/haar/J=2/sqtwolog/soft/scale=1.0/magnitude_phase/support=12 | DFT taps: 12 | clip factor: 5.0
[Impulsive] framework: WPT/haar/J=2/sqtwolog/soft/scale=1.0/magnitude_phase/support=12 | DFT taps: 12 | clip factor: 5.0


The helper functions below are also used in Part B. `top_candidates` shows the five best-ranked candidates (mean values on the validation set, zero-forcing receiver); rank 1 is the selected configuration. `support_tuning` shows the validation NMSE of the winner for each delay-support length.

In [6]:
def top_candidates(grid, selection, n=5):
    """Best-ranked candidates on the validation set (BER, then EVM, then NMSE)."""
    df = pd.DataFrame(selection["candidate_scores"])
    df.insert(0, "configuration", [c.label().rsplit("/support", 1)[0] for c in grid])
    df = df.sort_values(["ber", "evm_percent", "nmse"], kind="mergesort").head(n)   # stable: ties keep grid order
    df.index = pd.RangeIndex(1, len(df) + 1, name="rank")
    df = df.rename(columns={"ber": "BER", "evm_percent": "EVM (%)", "nmse": "NMSE", "snr_gain_db": "SNR gain (dB)"})
    return df.round({"BER": 4, "EVM (%)": 1, "NMSE": 4, "SNR gain (dB)": 2})


def support_tuning(val, selection, support_taps):
    """Validation NMSE of the selected configuration for each delay-support length."""
    best = selection["framework"]
    nmse = [float(np.mean(core.plain_nmse(val, core.wavelet_estimate(val.h_ls, replace(best, support_taps=k)))))
            for k in support_taps]
    return pd.DataFrame([nmse], columns=pd.Index(support_taps, name="support taps"), index=["validation NMSE"]).round(4)


for reg, label in syn.REGIMES.items():
    print(f"[{label}]")
    display(top_candidates(syn.GRID, selections13[reg]))
    display(support_tuning(splits[reg][0], selections13[reg], syn.SUPPORT_TAPS))

[AWGN]


,configuration,BER,EVM (%),NMSE,SNR gain (dB)
rank,,,,,
1,WPT/haar/J=2/sqtwolog/soft/scale=1.0/magnitude...,0.1019,130.1,0.1407,1.12
2,WPT/haar/J=1/sqtwolog/soft/scale=1.0/magnitude...,0.1023,138.2,0.1407,1.08
3,WPT/sym4/J=1/sqtwolog/soft/scale=1.0/component,0.1027,122.4,0.1523,0.76
4,DWT/haar/J=2/sqtwolog/soft/scale=1.0/magnitude...,0.1027,129.9,0.1574,0.78
5,WPT/haar/J=2/sqtwolog/soft/scale=1.0/component,0.1028,147.6,0.1485,0.99


support taps,0,4,8,12,16,24,32
validation NMSE,0.1407,0.3992,0.1651,0.0891,0.093,0.0939,0.0939


[Impulsive]


,configuration,BER,EVM (%),NMSE,SNR gain (dB)
rank,,,,,
1,WPT/haar/J=2/sqtwolog/soft/scale=1.0/magnitude...,0.1063,130.9,0.1690,0.86
2,WPT/haar/J=2/sqtwolog/soft/scale=1.0/component,0.1070,138.9,0.1764,0.79
3,WPT/haar/J=3/sqtwolog/soft/scale=1.0/component,0.1072,149.9,0.1718,0.60
4,WPT/db4/J=2/sqtwolog/soft/scale=1.0/magnitude_...,0.1072,151.8,0.1717,0.38
5,WPT/haar/J=3/sqtwolog/soft/scale=1.0/magnitude...,0.1073,136.4,0.1674,0.55


support taps,0,4,8,12,16,24,32
validation NMSE,0.169,0.4037,0.1773,0.1107,0.1181,0.1195,0.1195


### A5. Held-out test performance (Table 13)

All methods use an MMSE receiver. The DFT, fixed-wavelet and framework rows also include two iterations of decision-directed refinement. The values are means over the 240 test frames of each regime.

In [7]:
metrics13 = {reg: syn.evaluate_on_test(*splits[reg], selections13[reg]) for reg in syn.REGIMES}
table13 = pd.DataFrame([row for reg, label in syn.REGIMES.items()
                        for row in syn.performance_table(label, metrics13[reg])])
table13

,Noise Regime,Method,BER,EVM (%),NMSE,SNR Gain (dB)
0,AWGN,Raw LS,0.1066,60.5,0.2063,0.00
1,AWGN,DFT tuned,0.0877,50.0,0.0715,8.27
2,AWGN,Fixed wavelet,0.0934,51.3,0.1206,7.61
3,AWGN,Proposed framework,0.0883,50.1,0.0646,8.38
4,Impulsive,Raw LS,0.1013,60.0,0.2208,0.00
5,Impulsive,DFT tuned,0.0828,48.8,0.0716,8.63
6,Impulsive,Fixed wavelet,0.0905,50.4,0.1466,7.57
7,Impulsive,Proposed framework,0.0826,48.8,0.0693,8.61


### A6. Significance tests (Wilcoxon signed-rank on per-frame BER, Holm-corrected)

The reference is the framework's wavelet estimate before decision-directed refinement. It is compared with each of the other 8 estimators, and the Holm correction is applied over those 8 comparisons. The full Holm family is shown first; in the method names, `_DD` marks an estimate with decision-directed refinement. The final table holds the two comparisons reported in the article.

In [8]:
def holm_family(metrics, ref="LS_framework_selected_wavelet"):
    """All comparisons of the Holm family for one regime."""
    others = [n for n in sorted(metrics) if n != ref]
    raw = [wilcoxon(metrics[n]["ber"], metrics[ref]["ber"]).pvalue for n in others]
    return pd.DataFrame({"framework vs": others,
                         "p (uncorrected)": pd.Series(raw).map("{:.1e}".format),
                         "p (Holm)": pd.Series(syn.holm(raw)).map("{:.1e}".format)})


for reg, label in syn.REGIMES.items():
    print(f"[{label}] Holm family ({len(metrics13[reg]) - 1} comparisons)")
    display(holm_family(metrics13[reg]))

sig13 = pd.DataFrame([row for reg, label in syn.REGIMES.items()
                      for row in syn.significance_table(label, metrics13[reg])])
sig13

[AWGN] Holm family (8 comparisons)


,framework vs,p (uncorrected),p (Holm)
0,LS_DFT_DD,9.3e-12,2.8e-11
1,LS_DFT_validation_tuned,2.7e-02,5.4e-02
2,LS_LMMSE_covariance,4.0e-41,3.2e-40
3,LS_delay_MAD_clipping,4.0e-41,3.2e-40
4,LS_fixed_wavelet,1.2e-18,6.1e-18
5,LS_fixed_wavelet_DD,1.0e-01,1.0e-01
6,LS_framework_selected_wavelet_DD,1.3e-13,5.1e-13
7,LS_no_denoising,7.6e-23,4.5e-22


[Impulsive] Holm family (8 comparisons)


,framework vs,p (uncorrected),p (Holm)
0,LS_DFT_DD,1.0e-09,3.1e-09
1,LS_DFT_validation_tuned,1.5e-03,3.1e-03
2,LS_LMMSE_covariance,4.2e-41,3.3e-40
3,LS_delay_MAD_clipping,6.2e-41,4.3e-40
4,LS_fixed_wavelet,1.2e-12,4.8e-12
5,LS_fixed_wavelet_DD,2.5e-01,2.5e-01
6,LS_framework_selected_wavelet_DD,1.1e-13,5.7e-13
7,LS_no_denoising,2.5e-17,1.5e-16


,Noise Regime,Comparison,p (Holm)
0,AWGN,framework vs fixed wavelet,6.1e-18
1,AWGN,framework vs raw LS,4.5e-22
2,Impulsive,framework vs fixed wavelet,4.8e-12
3,Impulsive,framework vs raw LS,1.5e-16


---
## Part B: measured IEEE 802.11n channels 

### B1. Load the measured channel pool

The pool holds 445 measured channel vectors over 56 subcarriers, from an Atheros 802.11n CSI tool. Each channel is normalised to unit average power.

In [9]:
pool = mea.load_pool()
print(f"{pool.shape[0]} channels x {pool.shape[1]} subcarriers; "
      f"mean power per channel = {np.mean(np.abs(pool) ** 2, axis=1).mean():.3f}")

445 channels x 56 subcarriers; mean power per channel = 1.000


### B2. Build the frames and split them

Each frame applies one randomly drawn measured channel to a comb-pilot QPSK OFDM symbol (14 pilots and 42 data subcarriers), then adds noise. There are 80 frames per SNR value. Each regime has its own seeded stream: seed 7 for AWGN and seed 11 for impulsive noise. The frames are split 40/60 as in A2.

In [10]:
splits14 = {reg: mea.split_frames(mea.build_frames(pool, reg, seed), reg)
            for reg, (label, seed) in mea.REGIMES.items()}
pd.DataFrame([{"noise regime": label, "data seed": seed,
               "validation frames": len(splits14[reg][0].snr_db),
               "test frames": len(splits14[reg][1].snr_db)}
              for reg, (label, seed) in mea.REGIMES.items()])

,noise regime,data seed,validation frames,test frames
0,AWGN,7,160,240
1,Impulsive,11,160,240


### B3. Select the configuration on the validation set

The candidate set contains every admissible combination of:

- transform: DWT, SWT, WPT;
- mother wavelet: haar, db4, sym4;
- level: 1 to 3;
- threshold rule: sqtwolog, rigrsure;
- shrinkage: soft, garrote;
- threshold scale: 0.5, 1.0.

I and Q are processed separately. The ranking and support tuning are the same as in A4. The number of DFT taps is tuned by validation NMSE.

In [11]:
grid14 = pd.DataFrame([asdict(c) for c in mea.GRID])
print(len(grid14), "candidate configurations. Number per transform and wavelet:")
grid14.groupby(["transform", "wavelet"]).size().unstack(fill_value=0)

216 candidate configurations. Number per transform and wavelet:


wavelet,db4,haar,sym4
transform,,,
DWT,24,24,24
SWT,24,24,24
WPT,24,24,24


In [12]:
selections14 = {}
for reg, (label, _) in mea.REGIMES.items():
    selections14[reg] = s = mea.select_on_validation(splits14[reg][0])
    print(f"[{label}] framework: {s['framework'].label()} | DFT taps: {s['dft_taps']}")
    display(top_candidates(mea.GRID, s))
    display(support_tuning(splits14[reg][0], s, mea.SUPPORT_TAPS))

[AWGN] framework: SWT/haar/J=3/sqtwolog/soft/scale=1.0/component/support=0 | DFT taps: 56


,configuration,BER,EVM (%),NMSE,SNR gain (dB)
rank,,,,,
1,SWT/haar/J=3/sqtwolog/soft/scale=1.0/component,0.0693,90.3,0.1535,1.17
2,SWT/sym4/J=3/sqtwolog/soft/scale=1.0/component,0.0697,92.7,0.1522,1.02
3,SWT/db4/J=3/sqtwolog/soft/scale=1.0/component,0.0698,103.3,0.1530,0.88
4,WPT/haar/J=1/sqtwolog/soft/scale=1.0/component,0.0700,96.0,0.1627,0.91
5,WPT/sym4/J=1/sqtwolog/soft/scale=1.0/component,0.0702,88.4,0.1635,0.89


support taps,0,4,8,14,20,28,56
validation NMSE,0.1535,0.3562,0.3693,0.3699,0.3697,0.3695,0.1535


[Impulsive] framework: SWT/haar/J=3/sqtwolog/soft/scale=1.0/component/support=0 | DFT taps: 56


,configuration,BER,EVM (%),NMSE,SNR gain (dB)
rank,,,,,
1,SWT/haar/J=3/sqtwolog/soft/scale=1.0/component,0.0499,71.1,0.1425,1.04
2,DWT/haar/J=3/sqtwolog/soft/scale=1.0/component,0.0501,70.8,0.1526,0.74
3,SWT/sym4/J=3/sqtwolog/soft/scale=1.0/component,0.0506,75.5,0.1375,0.91
4,SWT/db4/J=3/sqtwolog/soft/scale=1.0/component,0.0506,79.1,0.1364,0.76
5,DWT/sym4/J=3/sqtwolog/soft/scale=1.0/component,0.0507,79.5,0.1508,0.64


support taps,0,4,8,14,20,28,56
validation NMSE,0.1425,0.2849,0.3005,0.3007,0.3005,0.3004,0.1425


### B4. Held-out test performance (Real measured channel)

The receiver is the same as in A5: MMSE equalisation, with decision-directed refinement for the DFT, fixed-wavelet and framework rows.

In [13]:
metrics14 = {reg: mea.evaluate_on_test(splits14[reg][1], selections14[reg]) for reg in mea.REGIMES}
table14 = pd.DataFrame([row for reg, (label, _) in mea.REGIMES.items()
                        for row in mea.performance_table(label, metrics14[reg])])
table14

,Noise Regime,Method,BER,EVM (%),NMSE,SNR Gain (dB)
0,AWGN,Raw LS,0.0713,52.4,0.2199,0.00
1,AWGN,DFT tuned,0.0712,52.1,0.2209,-0.60
2,AWGN,Fixed wavelet,0.0706,52.0,0.1866,-0.20
3,AWGN,Proposed framework,0.0670,51.3,0.1661,0.20
4,Impulsive,Raw LS,0.0541,47.9,0.2091,0.00
5,Impulsive,DFT tuned,0.0546,47.8,0.2130,-0.59
6,Impulsive,Fixed wavelet,0.0543,47.6,0.1872,-0.26
7,Impulsive,Proposed framework,0.0511,46.9,0.1747,0.12


### B5. Significance test (Wilcoxon signed-rank on per-frame channel NMSE, AWGN)

The test uses the framework's wavelet estimate before decision-directed refinement. It is compared with raw LS and with the fixed wavelet.

In [14]:
sig14 = pd.DataFrame(mea.significance_table(metrics14["awgn"]))
sig14

,Comparison,p
0,framework vs raw LS,4.0e-41
1,framework vs fixed wavelet,1.2e-39
